# Capítulo 7 — Aprendizado por Reforço Profundo

**Inteligência Artificial e Machine Learning Avançado para Defesa** — Prof. Cristiano Alves · Quantum Strategic AI

---

### 🎯 Objetivos do capítulo

Ao final deste notebook, você será capaz de:

1. **Formular** um problema de decisão sequencial como um **processo de decisão de Markov** (MDP) — estados, ações, recompensas, transições e fator de desconto — e descrever o laço agente–ambiente;
2. **Compreender** as **funções de valor** ($V$ e $Q$) e as **equações de Bellman** como o fundamento matemático do aprendizado por reforço;
3. **Aplicar** métodos baseados em **valor** — o *Q-learning* tabular, com exploração $\varepsilon$-gulosa, e sua extensão profunda, o **Deep Q-Network (DQN)**, com repetição de experiência e rede-alvo;
4. **Aplicar** métodos baseados em **política** — REINFORCE, ator-crítico (A2C) e **PPO** — e distinguir quando cada família é preferível;
5. **Reconhecer** as aplicações operacionais do reforço (decisão tática, otimização de trajetórias, controle de plataformas) e os seus riscos próprios — **má especificação da recompensa**, **lacuna simulação–realidade**, segurança — que tornam o **controle humano** indispensável quando um agente age.

> Os Módulos I e II construíram a **percepção**: extrair sentido — visual e textual — do fluxo bruto de dados. Mas perceber não é decidir, e decidir não é agir. O **Módulo III** trata da **decisão autônoma**, e com ele muda a natureza do problema. Até aqui, os modelos observavam e classificavam; a partir de agora, **um agente escolhe ações e, por meio delas, altera o mundo**. As apostas sobem na mesma medida: um classificador que erra produz um rótulo errado; **um agente que erra produz um ato errado**, com consequências que se propagam no tempo.
>
> O **aprendizado por reforço** (*Reinforcement Learning*, RL) é o paradigma em que um agente aprende a agir por **tentativa e recompensa**, sem um professor que lhe diga a resposta certa. E, porque aqui o modelo passa a atuar, a disciplina da obra deixa de ser metodologia de projeto e torna-se **requisito de segurança**: a má especificação de um objetivo não produz apenas um erro de *notebook* — produz um **comportamento autônomo indesejado**.

## Preparação do ambiente

As Seções 7.1 a 7.4 — MDP, Bellman, *Q-learning* e REINFORCE — executam com `numpy`, `matplotlib` e `torch`, já presentes no Colab, sobre **ambientes construídos no próprio notebook**. A Seção 7.5 usa `gymnasium` e `stable-baselines3` (PPO), instaladas em célula própria — e o treino do agente de pouso leva **~20 minutos em CPU** (o texto indica como encurtá-lo para um primeiro contato).

Como sempre, fixamos a **semente aleatória**: *dados os mesmos dados, os mesmos resultados*.

In [ ]:
# Preparacao do ambiente: importacoes e sementes de reprodutibilidade
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

SEMENTE = 0
np.random.seed(SEMENTE)
torch.manual_seed(SEMENTE)

print(f"numpy      {np.__version__}")
print(f"matplotlib {matplotlib.__version__}")
print(f"pytorch    {torch.__version__}")
print("Ambiente pronto.")

---
## 7.1 O problema do aprendizado por reforço

O RL formaliza a interação entre um **agente** e um **ambiente**. A cada instante $t$, o agente observa um **estado** $s_t$, escolhe uma **ação** $a_t$, e o ambiente responde com uma **recompensa** $r_{t+1}$ e um novo estado $s_{t+1}$. O agente repete esse laço buscando maximizar **não a recompensa imediata, mas a recompensa acumulada** ao longo do tempo.

Essa é uma forma de aprendizado distinta das anteriores. No supervisionado (Capítulos 2 e 3), cada exemplo vinha com o rótulo correto — um sinal **instrutivo**. No reforço, o agente recebe apenas um sinal **avaliativo** — *quão boa foi a consequência* —, muitas vezes **atrasado**: uma ação decisiva pode só ser recompensada muitos passos depois. Duas dificuldades daí decorrem: a **atribuição de crédito** (*que ação, na sequência, foi responsável pelo resultado?*) e a **exploração** (*para descobrir boas ações, o agente precisa experimentar as que ainda não conhece*).

Formalmente, o objetivo é maximizar o **retorno** — a soma descontada das recompensas futuras:

$$G_t = r_{t+1} + \gamma\, r_{t+2} + \gamma^2 r_{t+3} + \cdots = \sum_{k=0}^{\infty} \gamma^k r_{t+k+1},$$

onde o **fator de desconto** $\gamma \in [0, 1)$ pondera o presente contra o futuro: próximo de 0, o agente é **imediatista**; próximo de 1, é **previdente**. O comportamento do agente é descrito por uma **política** $\pi(a \mid s)$ — a probabilidade de escolher a ação $a$ no estado $s$.

> **🛡️ Contexto de defesa** — O reforço é o paradigma natural para a **decisão sequencial sob incerteza**, onde a consequência de um ato se desdobra no tempo: manobra tática, alocação dinâmica de recursos, otimização de trajetórias e controle de plataformas. São problemas em que não há um "rótulo correto" por passo — só o resultado, avaliado ao final. Mas é justamente por isso que, em defesa, o RL exige a cautela redobrada deste capítulo: **o agente otimiza o que se lhe pede, não o que se pretende** — e a distância entre as duas coisas é onde mora o risco.

## 7.2 Processos de decisão de Markov

O arcabouço matemático do RL é o **processo de decisão de Markov** (MDP), definido pela quíntupla $(\mathcal{S}, \mathcal{A}, P, R, \gamma)$: estados, ações, dinâmica de transição $P(s' \mid s, a)$, recompensa $R(s, a)$ e desconto. A **propriedade de Markov** afirma que o futuro depende apenas do estado presente, não de toda a história: *o estado resume tudo o que importa para decidir*.

### 7.2.1 Funções de valor

$$V^\pi(s) = \mathbb{E}_\pi\!\left[G_t \mid s_t = s\right], \qquad Q^\pi(s, a) = \mathbb{E}_\pi\!\left[G_t \mid s_t = s,\, a_t = a\right].$$

$V$ mede *quão bom é estar em um estado*; $Q$, *quão boa é uma ação em um estado* — a segunda é a mais útil para decidir.

### 7.2.2 As equações de Bellman

O que torna o problema tratável é a **estrutura recursiva** do valor. A política ótima satisfaz a **equação de otimalidade de Bellman**:

$$Q^*(s, a) = \sum_{s', r} P(s', r \mid s, a)\left[r + \gamma \max_{a'} Q^*(s', a')\right],$$

e, conhecido $Q^*$, agir de forma ótima é imediato: $\pi^*(s) = \arg\max_a Q^*(s, a)$. **Todo o aprendizado por reforço pode ser lido como o esforço de estimar $Q^*$ (ou diretamente $\pi^*$) a partir da experiência, sem conhecer de antemão a dinâmica $P$ do ambiente.**

Para trabalhar, construímos um **mundo em grade** com um dilema embutido — um objetivo **pequeno e próximo** e outro **grande e distante** (o fator de desconto decidirá entre eles, como verá o Exercício 7.1):

In [ ]:
# Mundo em grade 4x4 com dois objetivos: +3 (perto) e +10 (longe)
LADO = 4
n_estados, n_acoes = LADO * LADO, 4          # acoes: cima baixo esq dir
OBJETIVO_PERTO, R_PERTO = 3, 3.0             # canto sup. direito (3 passos)
OBJETIVO_LONGE, R_LONGE = 15, 10.0           # canto inf. direito (6 passos)
R_PASSO = -0.1                               # custo de cada movimento
N_ACOES_GRADE = 4    # constante da grade (a Secao 7.4 redefinira n_acoes)
MOVIMENTOS = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}

def ambiente_passo(s, a):
    # Dinamica do mundo em grade: devolve (novo_estado, recompensa, fim).
    linha, coluna = divmod(s, LADO)
    dl, dc = MOVIMENTOS[a]
    linha = min(max(linha + dl, 0), LADO - 1)
    coluna = min(max(coluna + dc, 0), LADO - 1)
    s_novo = linha * LADO + coluna
    if s_novo == OBJETIVO_PERTO:
        return s_novo, R_PERTO, True
    if s_novo == OBJETIVO_LONGE:
        return s_novo, R_LONGE, True
    return s_novo, R_PASSO, False

def desenha_grade(eixo, valores=None, titulo=""):
    if valores is not None:
        eixo.imshow(valores.reshape(LADO, LADO), cmap="viridis")
    for s in range(n_estados):
        l, c = divmod(s, LADO)
        rotulo = {0: "INICIO", OBJETIVO_PERTO: "+3", OBJETIVO_LONGE: "+10"}
        if s in rotulo:
            eixo.text(c, l, rotulo[s], ha="center", va="center",
                      color="white", fontsize=12, weight="bold")
    eixo.set_xticks([]); eixo.set_yticks([])
    eixo.set_title(titulo)

fig, eixo = plt.subplots(figsize=(4.6, 4.6))
desenha_grade(eixo, np.zeros(n_estados),
              "O dilema: +3 a 3 passos ou +10 a 6 passos?")
plt.tight_layout(); plt.show()
print("O agente parte do canto superior esquerdo (estado 0).")
print("Cada passo custa -0.1; os dois objetivos encerram o episodio.")

In [ ]:
# A equacao de Bellman em acao: iteracao de valor (dinamica CONHECIDA)
def iteracao_de_valor(gama=0.95, tolerancia=1e-8):
    # Programacao dinamica classica: aplica o operador de Bellman ate
    # convergir. Exige conhecer a dinamica P -- o que o RL nao exige.
    Q_estrela = np.zeros((n_estados, n_acoes))
    while True:
        Q_novo = np.zeros_like(Q_estrela)
        for s in range(n_estados):
            if s in (OBJETIVO_PERTO, OBJETIVO_LONGE):
                continue                      # estados terminais: valor 0
            for a in range(n_acoes):
                s_prox, r, fim = ambiente_passo(s, a)
                futuro = 0.0 if fim else np.max(Q_estrela[s_prox])
                Q_novo[s, a] = r + gama * futuro
        if np.max(np.abs(Q_novo - Q_estrela)) < tolerancia:
            return Q_novo
        Q_estrela = Q_novo

Q_estrela = iteracao_de_valor(gama=0.95)
V_estrela = Q_estrela.max(axis=1)

SETAS = {0: (0, -0.3), 1: (0, 0.3), 2: (-0.3, 0), 3: (0.3, 0)}
fig, eixo = plt.subplots(figsize=(5.2, 5.2))
desenha_grade(eixo, V_estrela, "V* e a politica otima (iteracao de valor)")
for s in range(n_estados):
    if s in (OBJETIVO_PERTO, OBJETIVO_LONGE):
        continue
    l, c = divmod(s, LADO)
    dx, dy = SETAS[int(np.argmax(Q_estrela[s]))]
    eixo.arrow(c, l, dx, dy, head_width=0.12, color="white")
plt.tight_layout(); plt.show()
print("V*(inicio) =", round(float(V_estrela[0]), 3),
      "| com gama = 0.95, o otimo e ir ao objetivo GRANDE.")

**Observe:** a iteração de valor aplicou a equação de Bellman **até convergir** — mas precisou **conhecer a dinâmica** do ambiente (pudemos consultar `ambiente_passo` à vontade, sem "viver" as transições). O aprendizado por reforço resolve o mesmo problema **sem esse privilégio**: apenas interagindo.

---
## 7.3 Métodos baseados em valor: Q-learning e DQN

### 7.3.1 Q-learning

O **Q-learning** aprende $Q^*$ por atualizações incrementais a partir de transições observadas, **sem modelo da dinâmica**. A cada passo $(s, a, r, s')$, ajusta a estimativa na direção do **alvo de Bellman amostrado**:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[\underbrace{r + \gamma \max_{a'} Q(s', a')}_{\text{alvo}} - Q(s, a)\right],$$

onde $\alpha$ é a taxa de aprendizado. Sob condições brandas, esse processo **converge para $Q^*$**. Resta o dilema entre **exploração e explotação**: usar sempre a ação de maior valor conhecido (*explotar*) impede descobrir ações melhores ainda não testadas (*explorar*). A solução mais simples é a **política $\varepsilon$-gulosa**: com probabilidade $\varepsilon$, agir ao acaso; caso contrário, agir de forma gulosa.

A **Listagem 7.1** implementa o método completo sobre o nosso mundo em grade — com **uma adaptação instrutiva** frente ao livro: cada episódio parte de um **estado aleatório** (*partidas exploratórias*). Partindo sempre do canto, o agente encontra o objetivo próximo, os episódios passam a terminar cedo — e o objetivo distante **jamais é descoberto** com $\varepsilon = 0{,}1$. A exploração é o gargalo do RL, e este mundo foi desenhado para expô-lo (o Exercício 7.2 volta à partida fixa para medir o efeito):

In [ ]:
# Listagem 7.1 - Q-learning tabular com exploracao epsilon-gulosa.
# (usa o ambiente_passo do mundo em grade definido acima)
Q = np.zeros((n_estados, n_acoes))

alfa, gama, epsilon = 0.1, 0.95, 0.1   # taxa, desconto, exploracao
rng = np.random.default_rng(SEMENTE)

def politica_epsilon_gulosa(s):
    if rng.uniform() < epsilon:
        return int(rng.integers(n_acoes))  # explora
    return int(np.argmax(Q[s]))            # explota (acao gulosa)

def partida_exploratoria():
    s = int(rng.integers(n_estados))       # estado inicial aleatorio
    while s in (OBJETIVO_PERTO, OBJETIVO_LONGE):
        s = int(rng.integers(n_estados))
    return s

for episodio in range(5000):
    s = partida_exploratoria()             # no livro: s = 0 (ver nota)
    for passo in range(100):
        a = politica_epsilon_gulosa(s)
        s_prox, r, fim = ambiente_passo(s, a)   # dinamica do ambiente

        # Atualizacao de Q-learning (alvo de Bellman amostrado):
        alvo = r + gama * (0.0 if fim else np.max(Q[s_prox]))
        Q[s, a] += alfa * (alvo - Q[s, a])

        s = s_prox
        if fim:
            break

# A politica otima e gulosa em relacao a Q aprendida.
politica = np.argmax(Q, axis=1)
print("Q-learning concluido: 5000 episodios de pura interacao.")

In [ ]:
# O que o agente aprendeu SEM conhecer a dinamica?
V_aprendido = Q.max(axis=1)

fig, eixos = plt.subplots(1, 2, figsize=(10.5, 4.8))
for eixo, V, titulo in [(eixos[0], V_estrela, "V* (Bellman, dinamica conhecida)"),
                        (eixos[1], V_aprendido, "V do Q-learning (so interacao)")]:
    desenha_grade(eixo, V, titulo)
    for s in range(n_estados):
        if s in (OBJETIVO_PERTO, OBJETIVO_LONGE):
            continue
        l, c = divmod(s, LADO)
        Qs = Q_estrela[s] if eixo is eixos[0] else Q[s]
        dx, dy = SETAS[int(np.argmax(Qs))]
        eixo.arrow(c, l, dx, dy, head_width=0.12, color="white")
plt.tight_layout(); plt.show()

def retorno_guloso(tabela_q, gama=0.95):
    # roda a politica gulosa a partir do inicio e mede o retorno
    s, total, fator = 0, 0.0, 1.0
    for _ in range(50):
        s, r, fim = ambiente_passo(s, int(np.argmax(tabela_q[s])))
        total += fator * r
        fator *= gama
        if fim:
            break
    return total

erro = float(np.mean(np.abs(V_aprendido - V_estrela)))
print(f"erro medio |V_aprendido - V*| = {erro:.3f}")
print(f"retorno da politica OTIMA (Bellman):     "
      f"{retorno_guloso(Q_estrela):+.3f}")
print(f"retorno da politica APRENDIDA (Q-learn): "
      f"{retorno_guloso(Q):+.3f}")
# (num mundo em grade ha VARIAS politicas otimas -- caminhos diferentes
#  de mesmo retorno; o que importa e o valor alcancado, nao as setas)

**Observe:** partindo do zero e interagindo às cegas, o *Q-learning* recuperou — apenas por tentativa e recompensa — praticamente a mesma função de valor e a mesma política que a programação dinâmica obteve com a dinâmica em mãos. É a promessa central do RL, cumprida no menor exemplo possível.

### 7.3.2 Deep Q-Networks (DQN)

O *Q-learning* tabular guarda um valor para **cada** par $(s, a)$ — inviável quando os estados são muitos ou contínuos, como uma imagem. O **Deep Q-Network (DQN)** resolve isso aproximando $Q(s, a; \theta)$ por uma **rede neural** (Capítulo 3). A ingenuidade, porém, desestabiliza o treino; dois artifícios o domam:

- A **repetição de experiência** (*experience replay*) armazena transições em uma memória e treina sobre minilotes amostrados ao acaso, **quebrando a correlação temporal** entre exemplos consecutivos;
- A **rede-alvo** (*target network*) — uma cópia da rede atualizada apenas periodicamente — fornece um **alvo estável** para a atualização, evitando que o modelo "persiga o próprio rabo".

Foi o DQN que, aprendendo a jogar dezenas de jogos de Atari **diretamente dos pixels**, tornou o RL profundo célebre.

> **✅ Boa prática** — A repetição de experiência e a rede-alvo **não são detalhes de implementação**: são o que separa um DQN que aprende de um que diverge. A lição geral é a do Capítulo 3 sob outra forma — a estabilidade do treino de redes profundas é frágil e exige técnica deliberada. Em RL, essa fragilidade é ainda maior, porque **os próprios dados de treino mudam à medida que a política muda**: o agente aprende sobre um alvo móvel que ele mesmo desloca.

---
## 7.4 Métodos baseados em política: REINFORCE, A2C e PPO

A segunda família dispensa o intermédio de $Q$ e otimiza **diretamente a política** $\pi_\theta(a \mid s)$, ajustando seus parâmetros por subida de gradiente sobre o retorno esperado.

O algoritmo fundamental é o **REINFORCE** (gradiente de política). Sua ideia é intuitiva: **aumentar a probabilidade das ações que levaram a retornos altos** e diminuir a das que levaram a retornos baixos. O gradiente do objetivo tem a forma

$$\nabla_\theta J(\theta) = \mathbb{E}_{\pi_\theta}\!\left[\nabla_\theta \log \pi_\theta(a \mid s) \cdot G_t\right],$$

em que $G_t$ é o retorno observado. A perda é, essencialmente, o **negativo do log-probabilidade ponderado pelo retorno** — a expressão acima, tornada concreta.

Para vê-lo funcionar, construímos um ambiente de **controle de plataforma à deriva** em 1D: uma correnteza empurra a plataforma, e o agente aplica impulsos (esquerda, nada, direita) para mantê-la próxima da posição de referência. A interface `reset()`/`step()` é a convenção universal dos ambientes de RL:

In [ ]:
# Ambiente de controle 1D: manter a plataforma proxima da referencia
class AmbienteDeriva:
    # Estado: [posicao, velocidade]. Acoes: 0=impulso a esquerda,
    # 1=nada, 2=impulso a direita. Recompensa maxima no centro.
    def __init__(self, semente=SEMENTE):
        self.rng = np.random.default_rng(semente)

    def reset(self):
        self.pos = float(self.rng.uniform(-1.5, 1.5))
        self.vel = 0.0
        self.passo = 0
        return np.array([self.pos, self.vel], dtype=np.float32)

    def step(self, acao):
        forca = (acao - 1) * 0.15
        correnteza = float(self.rng.normal(0, 0.02))
        self.vel = 0.9 * self.vel + forca + correnteza
        self.pos += self.vel
        self.passo += 1
        r = 1.0 - min(abs(self.pos) / 2.0, 1.5)   # +1 no centro
        fim = self.passo >= 60 or abs(self.pos) > 3.0
        return (np.array([self.pos, self.vel], dtype=np.float32),
                float(r), fim)

ambiente = AmbienteDeriva()
n_obs, n_acoes = 2, 3

# politica aleatoria, para referencia (a baseline de sempre)
rng = np.random.default_rng(SEMENTE)
retornos_aleatorios = []
for _ in range(200):
    obs, fim, total = ambiente.reset(), False, 0.0
    while not fim:
        obs, r, fim = ambiente.step(int(rng.integers(3)))
        total += r
    retornos_aleatorios.append(total)
print(f"retorno medio da politica ALEATORIA: "
      f"{np.mean(retornos_aleatorios):.1f} (maximo teorico ~60)")

In [ ]:
# Listagem 7.2 - REINFORCE (gradiente de politica) em PyTorch.
# Rede de politica: mapeia observacao -> logits sobre as acoes.
politica = nn.Sequential(
    nn.Linear(n_obs, 64), nn.ReLU(),
    nn.Linear(64, n_acoes),
)
otimizador = torch.optim.Adam(politica.parameters(), lr=1e-2)
gama = 0.99

def escolher_acao(obs):
    logits = politica(torch.tensor(obs, dtype=torch.float32))
    dist = torch.distributions.Categorical(logits=logits)
    acao = dist.sample()
    return acao.item(), dist.log_prob(acao)

historico_retornos = []
for episodio in range(600):
    log_probs, recompensas = [], []
    obs = ambiente.reset()
    fim = False
    while not fim:
        acao, logp = escolher_acao(obs)
        obs, r, fim = ambiente.step(acao)
        log_probs.append(logp)
        recompensas.append(r)

    # Retornos descontados G_t, calculados de tras para frente.
    G, retornos = 0.0, []
    for r in reversed(recompensas):
        G = r + gama * G
        retornos.insert(0, G)
    retornos = torch.tensor(retornos)
    retornos = (retornos - retornos.mean()) / (retornos.std() + 1e-8)

    # Gradiente de politica: -log_prob ponderado pelo retorno.
    perda = -torch.stack([lp * G for lp, G in zip(log_probs, retornos)]).sum()
    otimizador.zero_grad()
    perda.backward()
    otimizador.step()
    historico_retornos.append(sum(recompensas))

print(f"retorno medio (primeiros 50 episodios): "
      f"{np.mean(historico_retornos[:50]):.1f}")
print(f"retorno medio (ultimos 50 episodios):   "
      f"{np.mean(historico_retornos[-50:]):.1f}")

In [ ]:
# A curva de aprendizado e o comportamento antes/depois
media_movel = np.convolve(historico_retornos, np.ones(20) / 20, mode="valid")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.5, 4.2))
a1.plot(historico_retornos, alpha=0.3, label="retorno por episodio")
a1.plot(range(19, len(historico_retornos)), media_movel, lw=2,
        label="media movel (20)")
a1.axhline(np.mean(retornos_aleatorios), color="gray", ls=":",
           label="politica aleatoria")
a1.set_xlabel("episodio"); a1.set_ylabel("retorno")
a1.set_title("REINFORCE aprendendo por tentativa e recompensa")
a1.legend(fontsize=9)

# trajetorias da politica APRENDIDA (deterministica: acao mais provavel)
amb_avaliacao = AmbienteDeriva(semente=123)
for k in range(5):
    obs, fim, trajetoria = amb_avaliacao.reset(), False, []
    while not fim:
        with torch.no_grad():
            logits = politica(torch.tensor(obs, dtype=torch.float32))
        obs, r, fim = amb_avaliacao.step(int(logits.argmax()))
        trajetoria.append(obs[0])
    a2.plot(trajetoria, lw=1.5)
a2.axhline(0, color="k", lw=0.8)
a2.set_xlabel("passo"); a2.set_ylabel("posicao da plataforma")
a2.set_title("Politica aprendida: a deriva e corrigida para o centro")
plt.tight_layout(); plt.show()

**Observe:** o agente não recebeu um único exemplo de "ação correta" — apenas o sinal avaliativo da recompensa — e ainda assim as trajetórias convergem para a referência. Note também o **ruído** da curva de aprendizado: o REINFORCE puro sofre de **alta variância**.

O método **ator-crítico (A2C)** reduz essa variância combinando dois componentes: um **ator** (a política) e um **crítico** (uma função de valor) que fornece uma linha de base, substituindo o retorno bruto pela **vantagem** $A(s, a) = Q(s, a) - V(s)$ — *o quanto uma ação supera a média do estado*. Sobre essa base, o **PPO** (*Proximal Policy Optimization*) tornou-se o método de referência: estabiliza o aprendizado **limitando o quanto a política pode mudar** a cada atualização, por meio de um objetivo "recortado". Robusto e relativamente simples de ajustar, é hoje o cavalo de batalha do RL aplicado.

A escolha entre as famílias segue a natureza do problema: métodos de **valor** (DQN) convêm a **ações discretas**; métodos de **política** (PPO) lidam naturalmente com **ações contínuas** — ângulos de leme, potência, guinada —, o que os torna a opção usual em **controle de plataformas**.

---
## 7.5 Aplicação integrada: treinar e avaliar um agente de controle

Reunimos as peças em um problema de controle: **estabilizar e pousar uma plataforma simulada** — um análogo didático, e seguro, do controle de veículos autônomos. Treinamos um agente com **PPO** em simulação (Listagem 7.3) e — passo que este livro trata como **inegociável** — o avaliamos com rigor (Listagem 7.4), olhando não só o desempenho médio, mas a **cauda**: o pior caso e a taxa de falha.

> **📝 Nota** — Duas adaptações práticas: nas versões atuais da `gymnasium`, o ambiente chama-se `LunarLander-v3` (o `v2` do livro foi descontinuado); e o treino completo (300 mil passos) leva **~20 minutos em CPU** — para um primeiro contato, `total_timesteps=100_000` já mostra um agente competente (ajuste no código abaixo).

In [ ]:
# Dependencias da Secao 7.5 (no Colab, ~1-2 min)
%pip install -q swig
%pip install -q stable-baselines3 gymnasium[box2d]


In [ ]:
# Listagem 7.3 - Treino de um agente de controle com PPO (em simulacao).
import gymnasium as gym
from stable_baselines3 import PPO

# Ambiente de controle: estabilizacao e pouso simulados (proxy didatico
# e seguro para o controle de uma plataforma autonoma).
ambiente_ll = gym.make("LunarLander-v3")     # no livro: "LunarLander-v2"

# PPO: metodo de politica moderno e estavel (objetivo "recortado" que
# limita o quanto a politica muda a cada atualizacao).
modelo = PPO("MlpPolicy", ambiente_ll, verbose=0, seed=SEMENTE)
modelo.learn(total_timesteps=300_000)   # treino inteiramente em simulacao
modelo.save("agente_ppo")               # (~20 min em CPU; 100_000 para
print("Treino concluido.")              #  um primeiro teste mais rapido)

In [ ]:
# Listagem 7.4 - Avaliacao orientada a seguranca: media nao basta,
# a cauda decide.
modelo = PPO.load("agente_ppo")
ambiente_ll = gym.make("LunarLander-v3")

retornos, falhas = [], 0
for episodio in range(200):              # muitos episodios: a distribuicao
    obs, _ = ambiente_ll.reset(seed=episodio)
    total, fim, trunc = 0.0, False, False
    while not (fim or trunc):
        acao, _ = modelo.predict(obs, deterministic=True)
        obs, r, fim, trunc, _ = ambiente_ll.step(acao)
        total += r
    retornos.append(total)
    if total < 0:                        # criterio de falha do episodio
        falhas += 1

retornos = np.array(retornos)
# NAO basta a media: a decisao operacional exige a CAUDA (pior caso).
print(f"Retorno medio: {retornos.mean():.1f}")
print(f"Pior 5%:       {np.percentile(retornos, 5):.1f}")
print(f"Taxa de falha: {falhas / len(retornos):.1%}")

In [ ]:
# A distribuicao completa: onde vive o risco
plt.figure(figsize=(7.5, 4.2))
plt.hist(retornos, bins=30, color="tab:blue", alpha=0.8)
plt.axvline(retornos.mean(), color="k", lw=2, label="media")
plt.axvline(np.percentile(retornos, 5), color="tab:red", lw=2, ls="--",
            label="pior 5%")
plt.axvline(0, color="gray", ls=":", label="limiar de falha")
plt.xlabel("retorno do episodio"); plt.ylabel("episodios")
plt.title("Media alta nao garante seguranca: a cauda esquerda decide")
plt.legend(); plt.tight_layout(); plt.show()

**Observe:** a distribuição conta a história que a média esconde — e, **quando o agente age no mundo, é o pior caso que determina se o emprego é seguro**.

Dois riscos, próprios do RL, atravessam esse exemplo:

- A **má especificação da recompensa**: o agente otimiza **exatamente o que se recompensa**, e não o que se pretendia. Uma recompensa mal desenhada produz ***reward hacking*** — comportamentos que maximizam o número, mas subvertem a intenção (um agente instruído a "minimizar o tempo até o alvo" pode aprender a ignorar a segurança).
- A **lacuna simulação–realidade** (*sim-to-real gap*): agentes treinados em simulação, onde quase todo RL ocorre, **degradam-se no mundo real** — uma forma aguda do deslocamento de domínio dos Capítulos 4 e 6.

> **🛡️ Contexto de defesa** — O controle de plataformas autônomas por aprendizado é **tecnicamente contínuo** com o controle de sistemas de armas autônomos — a diferença está no *que* a plataforma faz, não em *como* aprende a agir. Essa continuidade é precisamente o que confere gravidade à discussão do Capítulo 10. Um agente de RL que decide e atua sem controle humano significativo transfere a uma **função de recompensa** — e a um comportamento emergente, difícil de prever e auditar — uma responsabilidade que, em defesa, **não pode ser delegada**. O reforço amplia o que uma máquina consegue fazer; **não altera quem deve responder pelo que ela faz**.

A célula abaixo produz um caso de *reward hacking* **de propósito**, no nosso mundo em grade — para que você veja o mecanismo do risco, não apenas leia sobre ele:

In [ ]:
# Reward hacking ao vivo: o proxy que subverte a intencao
# Intencao: "chegue ao objetivo". Proxy adicionado: "+0.3 por passo
# operacional" (parecia inofensivo -- recompensa 'atividade').
def ambiente_passo_proxy(s, a, bonus_proxy=0.3):
    s_novo, r, fim = ambiente_passo(s, a)
    if not fim:
        r += bonus_proxy          # recompensa por "manter-se operando"
    return s_novo, r, fim

def treina_e_mede(funcao_passo, episodios=5000, gama=0.99):
    Qh = np.zeros((n_estados, N_ACOES_GRADE))
    rng = np.random.default_rng(SEMENTE)
    for _ in range(episodios):
        s = int(rng.integers(n_estados))          # partida exploratoria
        while s in (OBJETIVO_PERTO, OBJETIVO_LONGE):
            s = int(rng.integers(n_estados))
        for _ in range(100):
            a = (int(rng.integers(N_ACOES_GRADE)) if rng.uniform() < 0.1
                 else int(np.argmax(Qh[s])))
            s_prox, r, fim = funcao_passo(s, a)
            alvo = r + gama * (0.0 if fim else np.max(Qh[s_prox]))
            Qh[s, a] += 0.1 * (alvo - Qh[s, a])
            s = s_prox
            if fim:
                break
    # avaliacao: a politica gulosa chega a algum objetivo em 100 passos?
    chegadas = 0
    for _ in range(100):
        s, chegou = 0, False
        for _ in range(100):
            s, _, fim = funcao_passo(s, int(np.argmax(Qh[s])))
            if fim:
                chegou = True
                break
        chegadas += chegou
    return chegadas

print("politica treinada com a recompensa ORIGINAL:")
print(f"  chega a um objetivo em {treina_e_mede(ambiente_passo)}% dos episodios")
print("politica treinada com o proxy de 'atividade' (+0.3/passo):")
print(f"  chega a um objetivo em "
      f"{treina_e_mede(ambiente_passo_proxy)}% dos episodios")
print("\nO agente do proxy descobriu que vagar para sempre paga mais do")
print("que concluir a missao: otimizou o numero, subverteu a intencao.")

**Observe:** bastou um bônus aparentemente inofensivo — "+0,3 por passo operacional" — para o ótimo deixar de ser *concluir a missão* e passar a ser *vagar indefinidamente*: com $\gamma = 0{,}99$, o fluxo perpétuo de bônus vale mais do que qualquer objetivo terminal. **O agente não errou: otimizou exatamente o que foi pedido.** O erro foi de quem especificou a recompensa — e é por isso que ela deve ser projetada e testada com o mesmo rigor do modelo (Exercício 7.5).

---
## 7.6 O caminho à frente

Este capítulo deu ao repertório da obra a capacidade de **decidir e agir**: do MDP e das equações de Bellman aos métodos de valor (DQN) e de política (PPO), e às suas aplicações — e riscos — no controle autônomo. Mas um sistema que age é também um sistema que **pode ser atacado**. O **Capítulo 8** fecha o Módulo III com a **robustez adversarial**: as vulnerabilidades de todos os modelos vistos até aqui — classificadores, detectores, LLMs e agentes de RL — a manipulações deliberadas (envenenamento de dados, exemplos adversariais, ataques físicos) e as defesas correspondentes.

A disciplina segue, agora sob as maiores apostas do livro: comparar contra uma *baseline*, **medir pela cauda e não só pela média**, validar no emprego real e manter o **controle humano** sobre a decisão. Quando o modelo passou a agir, essas exigências deixaram de ser boas práticas e tornaram-se **condições de segurança**.

## 📋 Resumo do capítulo

- No **aprendizado por reforço**, um agente aprende a agir por tentativa e recompensa, maximizando o retorno descontado $G_t = \sum_k \gamma^k r_{t+k+1}$. O sinal é **avaliativo e atrasado**, o que impõe a *atribuição de crédito* e a *exploração*.

- O problema formaliza-se como um **MDP** $(\mathcal{S}, \mathcal{A}, P, R, \gamma)$; as funções de valor $V$ e $Q$ e as **equações de Bellman** são seu fundamento, e a política ótima é gulosa em relação a $Q^*$.

- Os métodos baseados em **valor** estimam $Q^*$: o ***Q-learning*** tabular, com exploração $\varepsilon$-gulosa, e o **DQN**, que o estende a estados complexos com rede neural, **repetição de experiência** e **rede-alvo**.

- Os métodos baseados em **política** otimizam $\pi_\theta$ diretamente: **REINFORCE** (gradiente de política, alta variância), **A2C** (ator-crítico, com a vantagem $A = Q - V$) e **PPO** (atualização estável e recortada), preferível em **ações contínuas** de controle.

- As aplicações — decisão tática, otimização de trajetórias, controle de plataformas — trazem riscos próprios: **má especificação da recompensa** (*reward hacking*) e **lacuna simulação–realidade**.

- A avaliação de um agente que age deve olhar a **cauda** (pior caso, taxa de falha), não só a média; e o **controle humano significativo** é indispensável, pela continuidade entre controle autônomo e sistemas de armas (Capítulo 10).

## ⚠️ Armadilhas comuns

1. **Especificar mal a recompensa.** O agente otimiza o que se recompensa, não o que se pretende. Recompensas mal desenhadas geram *reward hacking* — comportamentos que maximizam o número e subvertem a intenção. Projete e teste a recompensa com o mesmo cuidado que o modelo.

2. **Ignorar a lacuna simulação–realidade.** Um agente excelente em simulação pode falhar no mundo real. Valide nas condições de emprego e trate a transferência como um deslocamento de domínio a mitigar.

3. **Avaliar pela média.** Um bom retorno médio pode esconder falhas catastróficas na cauda. Reporte o pior caso e a taxa de falha — quando o agente age, é a cauda que decide a segurança.

4. **Explorar de forma insuficiente ou insegura.** Pouca exploração trava o agente em políticas medíocres; exploração irrestrita no mundo real pode ser perigosa. Calibre a exploração e prefira treinar em simulação.

5. **Usar RL onde um controlador mais simples bastaria.** O reforço é caro, instável e difícil de auditar. Para muitos problemas de controle, um método clássico é mais previsível e seguro — exija que o RL prove sua vantagem.

6. **Delegar a decisão a um agente autônomo.** O comportamento emergente de um agente de RL é difícil de prever e auditar. Mantenha o **controle humano significativo** sobre a decisão de agir, sobretudo pela proximidade com sistemas de armas autônomos (Capítulo 10).

---
## 🧭 Exercícios

Classificação: **Essencial** (fixação) · **Tático** (aplicação) · **Estratégico** (extensão criativa). Os exercícios de código têm células preparadas abaixo; os dissertativos podem ser respondidos em células de texto no próprio notebook.

### Essencial

**Exercício 7.1** — Execute a célula abaixo (Listagem 7.1 sobre o mundo em grade) variando o fator de desconto $\gamma$ em $\{0{,}5;\ 0{,}9;\ 0{,}99\}$. Relate **como a política aprendida muda** — a qual objetivo o agente passa a ir — e explique, em duas linhas, por que um $\gamma$ baixo produz um agente imediatista.

In [ ]:
# Exercicio 7.1 - o fator de desconto decide entre o perto e o longe
def treina_q(gama, epsilon=0.1, episodios=5000):
    Qe = np.zeros((n_estados, N_ACOES_GRADE))
    rng = np.random.default_rng(SEMENTE)
    for _ in range(episodios):
        s = int(rng.integers(n_estados))          # partida exploratoria
        while s in (OBJETIVO_PERTO, OBJETIVO_LONGE):
            s = int(rng.integers(n_estados))
        for _ in range(100):
            a = (int(rng.integers(N_ACOES_GRADE)) if rng.uniform() < epsilon
                 else int(np.argmax(Qe[s])))
            s_prox, r, fim = ambiente_passo(s, a)
            alvo = r + gama * (0.0 if fim else np.max(Qe[s_prox]))
            Qe[s, a] += 0.1 * (alvo - Qe[s, a])
            s = s_prox
            if fim:
                break
    return Qe

def destino_da_politica(Qe):
    s = 0
    for _ in range(50):
        s, _, fim = ambiente_passo(s, int(np.argmax(Qe[s])))
        if fim:
            return "+3 (PERTO)" if s == OBJETIVO_PERTO else "+10 (LONGE)"
    return "nenhum (vagou)"

for gama in [0.5, 0.9, 0.99]:      # <--- ALTERE e explore outros valores
    Qe = treina_q(gama)
    print(f"gama = {gama:4.2f} -> politica gulosa vai ao objetivo "
          f"{destino_da_politica(Qe)}")

# Sua resposta (2 linhas):
# Com gama baixo, a recompensa distante vale gama^6 * 10 = ... , menos
# que a proxima (gama^3 * 3 = ...), porque ...

**Exercício 7.2** — Agora com **partida fixa** no canto (como no livro), varie $\varepsilon$ em $\{0{,}0;\ 0{,}1;\ 0{,}5\}$ e observe **duas métricas**: a qualidade da política final e o retorno médio obtido **durante** o treino (desempenho *online*). Explique o que ocorre com $\varepsilon = 0$ (sem exploração) e por que a exploração excessiva também tem custo — relacionando as duas métricas ao dilema entre exploração e explotação.

In [ ]:
# Exercicio 7.2 - o dilema exploracao vs. explotacao (partida FIXA)
def treina_q_online(gama, epsilon, episodios=5000):
    # treina com partida fixa e mede tambem o retorno DURANTE o treino
    Qe = np.zeros((n_estados, N_ACOES_GRADE))
    rng = np.random.default_rng(SEMENTE)
    retornos_online = []
    for _ in range(episodios):
        s, total = 0, 0.0
        for _ in range(100):
            a = (int(rng.integers(N_ACOES_GRADE)) if rng.uniform() < epsilon
                 else int(np.argmax(Qe[s])))
            s_prox, r, fim = ambiente_passo(s, a)
            alvo = r + gama * (0.0 if fim else np.max(Qe[s_prox]))
            Qe[s, a] += 0.1 * (alvo - Qe[s, a])
            total += r
            s = s_prox
            if fim:
                break
        retornos_online.append(total)
    return Qe, float(np.mean(retornos_online))

def retorno_da_politica_gulosa(Qe, gama=0.95):
    s, total, fator = 0, 0.0, 1.0
    for _ in range(50):
        s, r, fim = ambiente_passo(s, int(np.argmax(Qe[s])))
        total += fator * r
        fator *= gama
        if fim:
            return total
    return total

print(f"{'eps':>5s} | {'politica final':>14s} | {'destino':>12s} | online")
for eps in [0.0, 0.1, 0.5]:        # <--- ALTERE e explore outros valores
    Qe, online = treina_q_online(gama=0.95, epsilon=eps)
    print(f"{eps:5.1f} | {retorno_da_politica_gulosa(Qe):14.2f} | "
          f"{destino_da_politica(Qe):>12s} | {online:6.2f}")

# Sua resposta (poucas linhas):
# 1) Com epsilon = 0, o agente trava no primeiro caminho que da certo: ...
# 2) Epsilon alto encontra a politica MELHOR aqui, mas o retorno online
#    cai porque ... (e agir mal DURANTE o aprendizado custa, no mundo real)
# 3) O dilema exploracao/explotacao consiste em ...

**Exercício 7.3** *(dissertativo)* — A partir da equação de otimalidade de Bellman, mostre por escrito que, conhecido $Q^*$, a política gulosa $\pi^*(s) = \arg\max_a Q^*(s, a)$ é ótima. Explique, em uma frase, por que estimar $Q^*$ resolve, em princípio, o problema de decisão.

*Responda em uma célula de texto abaixo.*

### Tático

**Exercício 7.4** — Treine o agente da Listagem 7.3 e, com a Listagem 7.4, compare o **retorno médio**, o **pior 5%** e a **taxa de falha** em 200 episódios. Discuta, em poucas linhas, por que um agente com bom retorno médio pode ainda ser **inseguro** para emprego real. *(Requer a Seção 7.5 executada; use a célula abaixo para organizar a resposta.)*

In [ ]:
# Exercicio 7.4 - a avaliacao orientada a cauda, em resumo
# (reusa o array `retornos` da Listagem 7.4)
print(f"retorno medio:  {retornos.mean():8.1f}")
print(f"mediana:        {np.median(retornos):8.1f}")
print(f"pior 5%:        {np.percentile(retornos, 5):8.1f}")
print(f"pior episodio:  {retornos.min():8.1f}")
print(f"taxa de falha:  {(retornos < 0).mean():8.1%}")

# Sua resposta (poucas linhas):
# 1) A media esconde ... porque ...
# 2) Para um emprego real, o numero que decide e ... porque quando o
#    agente AGE, uma falha rara significa ...

**Exercício 7.5** — A célula abaixo reproduz o *reward hacking* da Seção 7.5 com o bônus-proxy ajustável. Varie `bonus_proxy` em $\{0{,}0;\ 0{,}05;\ 0{,}3\}$, encontre (aproximadamente) o valor em que o comportamento **vira**, e proponha uma correção da recompensa que preserve a intenção ("chegue ao objetivo") mesmo com o bônus presente.

In [ ]:
# Exercicio 7.5 - induzindo e corrigindo o reward hacking
for bonus in [0.0, 0.05, 0.3]:     # <--- ALTERE: onde o comportamento vira?
    passo_fn = lambda s, a, b=bonus: ambiente_passo_proxy(s, a, bonus_proxy=b)
    chegadas = treina_e_mede(passo_fn)
    print(f"bonus_proxy = {bonus:4.2f} -> chega ao objetivo em "
          f"{chegadas}% dos episodios")

# Sua resposta (poucas linhas):
# 1) O comportamento vira aproximadamente em bonus = ... porque o fluxo
#    perpetuo bonus/(1-gama) = ... supera o valor do objetivo.
# 2) Correcao proposta (ex.: limitar o horizonte, descontar o bonus no
#    terminal, penalizar o tempo): ...

**Exercício 7.6** *(dissertativo)* — Considere o problema de **otimização de trajetória** de um veículo autônomo entre dois pontos, evitando zonas de risco. Formule-o como um MDP: defina estados, ações, recompensa e desconto, e indique se você o atacaria com um método de **valor** (DQN) ou de **política** (PPO), justificando pela natureza das ações.

*Responda em uma célula de texto abaixo.*

### Estratégico

**Exercício 7.7** — Em um texto técnico (até três páginas), discuta os **riscos de segurança** do emprego de agentes de RL em plataformas autônomas de defesa — má especificação da recompensa, lacuna simulação–realidade e comportamento emergente — e proponha um **protocolo de avaliação e de controle humano** que os mitigue antes de qualquer emprego real. Relacione o argumento ao Capítulo 10.

**Exercício 7.8** — Projete, em diagrama, uma arquitetura de **decisão tática assistida por RL** em que o agente **recomenda, mas não executa**, ações — mantendo o operador humano no ciclo de decisão. Especifique os pontos de intervenção humana, a métrica de avaliação orientada à cauda e as salvaguardas contra a confiança cega na recomendação do agente (viés de automação).

**Exercício 7.9** — Escolha um ambiente de controle público (por exemplo, da biblioteca Gymnasium) e conduza um **estudo completo**: treine um agente com PPO, avalie-o com atenção à cauda, compare-o a uma *baseline* (um controlador simples ou uma política aleatória) e redija um miniartigo de até três páginas documentando a recompensa empregada, os resultados com sua variância, os **modos de falha** observados e os limites de transferência para um cenário operacional real.

---

*Fim do Capítulo 7 — no Capítulo 8, a robustez adversarial: o que acontece quando os modelos deste livro são atacados.*